In [ ]:
"""
Task:
    Extract complete self-reported employment histories for Revelio users linked to a
    PatentsView inventor.

Inputs:
(a) Files/WenzhiW/B01_ConstructAnalysisSample/Inventors_USPTO_UserID.parquet
(b) user_positions (Microsoft Fabric table)

Outputs:
(a) Files/WenzhiW/B01_ConstructAnalysisSample/Inventors_USPTO_UserPositions
(b) Files/WenzhiW/B01_ConstructAnalysisSample/Inventors_USPTO_UserPositions.zip

Description of outputs:
(1) Data (a) is a multi-part Parquet dataset containing every row in user_positions for
    an inventor-linked user and every variable delivered by the source table.
(2) File (b) wraps the Parquet directory without recompressing its already compressed files.

Notes:
(1) Upload the output of D01 to the input path above before running this notebook.
(2) A broadcast left-semi join filters the large table in one pass and does not add columns.
(3) No date, occupation, geography, or nonmissing-value restriction is applied to positions.

Wang Wenzhi, with the help of Codex
Time: 2026-08-27
"""

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegralType, StringType


# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 0. Specify paths and define a schema helper
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


POSITION_TABLE = "user_positions"
INPUT_USER_IDS = (
    "Files/WenzhiW/B01_ConstructAnalysisSample/"
    "Inventors_USPTO_UserID.parquet"
)
OUTPUT_PARQUET = (
    "Files/WenzhiW/B01_ConstructAnalysisSample/"
    "Inventors_USPTO_UserPositions"
)
OUTPUT_WRITE_MODE = "overwrite"


def validate_required_columns(data, required_columns, dataset_name):
    """Fail from schema metadata when a required column is absent."""

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing required columns: {missing_columns}."
        )


spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


## Step 1. Prepare the small inventor-user filter

Only the uploaded ID file is materialized here. The large positions table is inspected through
schema metadata, and the small-side IDs are cast to its delivered `user_id` type so no
transformation is applied to the large table's join key.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 1. Validate schemas and materialize the small-side user list
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


uploaded_user_ids = spark.read.parquet(INPUT_USER_IDS)
validate_required_columns(uploaded_user_ids, ("user_id",), INPUT_USER_IDS)

user_positions_source = spark.read.table(POSITION_TABLE)
validate_required_columns(user_positions_source, ("user_id",), POSITION_TABLE)

position_columns = tuple(user_positions_source.columns)
position_schema = tuple(
    (field.name, field.dataType.simpleString())
    for field in user_positions_source.schema.fields
)
source_user_id_type = user_positions_source.schema["user_id"].dataType

if not isinstance(source_user_id_type, (IntegralType, StringType)):
    raise TypeError(
        "user_positions.user_id must be an integral or text field; found "
        f"{source_user_id_type.simpleString()}."
    )

uploaded_unique_user_count = (
    uploaded_user_ids.select("user_id")
    .where(F.col("user_id").isNotNull())
    .distinct()
    .count()
)

inventor_user_ids = (
    uploaded_user_ids.select(
        F.col("user_id").cast(source_user_id_type).alias("user_id")
    )
    .where(F.col("user_id").isNotNull())
    .dropDuplicates(("user_id",))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
inventor_user_count = inventor_user_ids.count()

if inventor_user_count == 0:
    inventor_user_ids.unpersist()
    raise ValueError("The uploaded inventor user list contains no usable user IDs.")
if inventor_user_count != uploaded_unique_user_count:
    inventor_user_ids.unpersist()
    raise ValueError(
        "Casting the uploaded user IDs to user_positions.user_id changed the "
        "number of unique IDs. Check the two source schemas."
    )

print(f"Validated {len(position_columns):,} variables in {POSITION_TABLE}.")
print(f"user_positions.user_id type: {source_user_id_type.simpleString()}.")
print(f"Prepared {inventor_user_count:,} unique inventor-linked users.")


## Step 2. Filter and write the complete employment histories

The left-semi join is a membership filter: it returns only columns from `user_positions`, so all
source variables and their delivered values are retained. Broadcasting the roughly user-level
filter avoids shuffling the much larger positions table. The write is the only action that scans
the large source table.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 2. Apply one broadcast left-semi join and write the result
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


inventor_positions = user_positions_source.join(
    F.broadcast(inventor_user_ids),
    on="user_id",
    how="left_semi",
).select(*position_columns)

if tuple(inventor_positions.columns) != position_columns:
    inventor_user_ids.unpersist()
    raise ValueError(
        "The semi-join output columns differ from the user_positions source columns."
    )

print("Physical plan for the positions filter:")
inventor_positions.explain(mode="simple")

try:
    inventor_positions.write.mode(OUTPUT_WRITE_MODE).parquet(OUTPUT_PARQUET)
    print(f"Multi-part Parquet dataset saved: {OUTPUT_PARQUET}.")
finally:
    inventor_user_ids.unpersist()


## Step 3. Validate the written subset

Validation rereads only the filtered Parquet output. The row/user aggregation projects the
single `user_id` column, avoiding a second scan of all columns in the raw positions table.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 3. Check the output schema and exact matched-user coverage
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


written_positions = spark.read.parquet(OUTPUT_PARQUET)
written_schema = tuple(
    (field.name, field.dataType.simpleString())
    for field in written_positions.schema.fields
)

if written_schema != position_schema:
    raise ValueError(
        "The written output does not preserve the source column names, order, and types."
    )

output_stats = written_positions.agg(
    F.count(F.lit(1)).cast("long").alias("employment_spell_count"),
    F.countDistinct("user_id").cast("long").alias("matched_user_count"),
).first()

employment_spell_count = output_stats["employment_spell_count"]
matched_user_count = output_stats["matched_user_count"]
unmatched_user_count = inventor_user_count - matched_user_count

if employment_spell_count == 0:
    raise ValueError("The written inventor employment-history dataset is empty.")
if matched_user_count > inventor_user_count:
    raise ValueError("The output contains more users than the uploaded inventor list.")

print("Inventor employment-history summary:")
print(f"Employment spells: {employment_spell_count:,}")
print(f"Inventor-linked users with at least one position: {matched_user_count:,}")
print(f"Uploaded users without a position: {unmatched_user_count:,}")
print(f"Variables retained from user_positions: {len(position_columns):,}")


## Step 4. Create a download-ready archive

Parquet already compresses its columns. `ZIP_STORED` bundles the distributed output without
spending compute on a second compression pass, and Zip64 supports archives above 4 GiB.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 4. Wrap the multi-part Parquet directory for local download
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


from pathlib import Path
from zipfile import ZIP_STORED, ZipFile


source_dir = Path(
    "/lakehouse/default/Files/WenzhiW/B01_ConstructAnalysisSample/"
    "Inventors_USPTO_UserPositions"
)
archive_path = source_dir.parent / "Inventors_USPTO_UserPositions.zip"

if not source_dir.is_dir():
    raise FileNotFoundError(f"Source directory not found: {source_dir}")

source_files = sorted(path for path in source_dir.rglob("*") if path.is_file())
if not source_files:
    raise FileNotFoundError(f"No files were written under: {source_dir}")

if archive_path.exists():
    archive_path.unlink()

with ZipFile(
    archive_path,
    mode="w",
    compression=ZIP_STORED,
    allowZip64=True,
) as archive:
    for source_file in source_files:
        relative_path = source_file.relative_to(source_dir).as_posix()
        archive.write(
            source_file,
            arcname=f"{source_dir.name}/{relative_path}",
        )

total_gib = sum(path.stat().st_size for path in source_files) / (1024**3)

print(f"Created: {archive_path}")
print(f"Archived files: {len(source_files):,}")
print(f"Uncompressed size: {total_gib:,.2f} GiB")
